# Survival analysis of MB
Including Chapman *et al.* 2023, 491 available medulloblastoma cases with ecDNA annotations. How well do ecDNA, MYC family amplification, and amplification (any gene, any topology) predict survival?

**KM on amplicon class**  
Clean separation of ecDNA+ chrAmp+, and nonamplified cases; but possibly confounded by MYC family or other covariates.

**Cox model on amplification status** does not satisfy the proportional hazards assumption, so we perform a **landmark split at 24mo.** w.r.t. ecDNA. Estimated hazards for 
- ecDNA (0-24] = 4.61, p<0.001.
- ecDNA (24-60] = 0.65, p=0.498.

**KM of amplified cases stratified by ecDNA+/-, MYC family amp +/-**
Significant separation of ecDNA+ MYC+ vs ecDNA+ MYC- (p=0.011) and chr+ MYC- (p=5.2e-06) but underpowered to compare vs. chr+ MYC+ (p=0.49) due to only n=6 examples of the latter, and only 2 events.  
Therefore, **Cox models including MYC family amp +/-** are underpowered.

In [ ]:
Sys.setenv(LANGUAGE = "en")

library(tidyverse)
library(readxl)
library(dplyr)
library(stringr)
library(naniar) #for replace with Nas function
library(survival)
library(coxme)
library(survminer)
library(RColorBrewer)
library(janitor)
library(gt)
library(gtsummary)
library(ggsurvfit)
library(extrafont)
library(svglite)

extrafont::font_import(pattern="Arial",prompt=FALSE)
extrafont::loadfonts()

# imports from external file
imports <- new.env()
source("survival-data-imports.R", local = imports)
plotting <- new.env()
source("survival-plots.R",local=plotting)

sessionInfo()

In [ ]:
## create an output directory if it doesn't exist
dir.create('out', showWarnings = FALSE)
## Global path variables
ST_PATH="../../data/Supplementary Tables.xlsx"
CHAPMAN2023_PATH='../../data/external/Chapman2023/41588_2023_1551_MOESM4_ESM.xlsx'

In [ ]:
load_chapman_2023_genes <- function(path){
    ## path: path to Chapman et al. 2023 supplementary tables.
    ## include_archer: whether to include the "Archer" subcohort. These should be included (they're ICGC samples), but were not until 2025-11-06.
    tbl <- read_tsv(path)

    format_ids <- function(id_vector){
        # format Archer IDs to match ICGC
        prefixes <- c("ICGC_", "MBRep", "MDT-AP")
        pattern <- paste0("^(", paste(prefixes, collapse = "|"), ")")
        id_vector <- case_when( 
            str_detect(id_vector, pattern) ~ id_vector,  # already has prefix → leave unchanged
            TRUE ~ paste0(
                "ICGC_",
                str_replace(id_vector, "^([A-Za-z]+)0*([0-9]+)$", "\\1\\2")  # remove leading zeros in number
            )
        )
        # drop _AA suffix
        id_vector <- str_remove(id_vector, "_AA$") 
        return(id_vector)
    }
    tbl = tbl %>% 
        mutate(sample_name = format_ids(sample_name)) %>%
        mutate(patient_id = sample_name)
    return (tbl)
}
load_genes <- function(path){
    genes = read_excel(path,sheet="5. Gene amplifications",col_types=c('text','text','text','text','numeric','text','logical')) %>% suppressWarnings
    samples = read_excel(path,'2. Biosamples',col_types=c('text','skip','text',rep('skip',12))) %>% suppressWarnings %>%
        rename(sample_name = biosample_id)
    return(genes %>% left_join(samples,by='sample_name'))
}
annotate_gene_amplifications <- function(pt_df, gene_df, genes){
    # TODO sample_name -> patient_id
    gene_df <- gene_df %>% 
        filter(gene %in% genes) %>%
        distinct(patient_id, gene) %>%
        mutate(value = TRUE) %>%
        pivot_wider(
            names_from = gene,
            values_from = value,
            values_fill = NA
        )
    return(pt_df %>% left_join(gene_df,by='patient_id'))
}
#load_genes(ST_PATH) %>% head
#load_chapman_2023_genes('../../data/external/Chapman2023/grch37_gene_list.tsv') %>% head

dd <- imports$load_survival_data(ST_PATH,CHAPMAN2023_PATH) %>%
  filter(str_detect(cancer_type, "MBL"))
dd %>% head

### KM of 491 MBL, stratified by amplicon class

KM pairwise log-rank tests:
```
                 chromosomal ecDNA 
ecDNA            0.0037      -     
no amplification 0.5736      0.0001
```

In [ ]:
formula = Surv(OS_months_5y, OS_status_5y) ~ amplicon_class
km = survfit2(formula=formula, data = dd)
plotting$km_plot(km) +
  geom_vline(xintercept = 24, linetype = 2, colour = "grey50")
plotting$save_ggplot("km_mb_5year")
logrank <- pairwise_survdiff(formula,dd,p.adjust.method="BH",rho=0)
logrank

### Fully parameterized Cox model, n=445
Hazard ratios:
```
Hazard ratio ecDNA: 2.79 ; p= 0.000581617250296626
Hazard ratio amplification: 0.9 ; p= 0.708680702195459
```
However, proportional hazards assumption is violated for ecDNA, age.  
`cox.zph` proportionality tests:
```
                   chisq df     p
ecDNA_status      5.7914  1 0.016
amp_status        0.0891  1 0.765
sex               1.9706  1 0.160
age_at_diagnosis  4.3949  1 0.036
cancer_subclass   2.5511  3 0.466
GLOBAL           17.8683  7 0.013
```

In [ ]:
# cox regression on amp and ecDNA
dmb = dd %>% 
    filter(cancer_subclass %in% c('WNT','SHH','G3','G4')) %>%
    mutate(cancer_subclass = factor(cancer_subclass))
dmb$cancer_subclass = relevel(dmb$cancer_subclass, ref = 'G4')

mb1 <- coxph(Surv(OS_months_5y, OS_status_5y) ~ ecDNA_status + amp_status + sex + age_at_diagnosis + cancer_subclass, data = dmb)
mb1
#plotting$cox_plot(mb1, dmb)
#plotting$save_ggplot("cox_mb_forest", width = 6, height = 4.5)
plotting$forest_coxph(mb1)

In [ ]:
#mb1$coefficients
print(paste("Hazard ratio ecDNA:",round(exp(mb1$coefficients[['ecDNA_statusecDNA+']]),2),"; p=",(summary(mb1)$coefficients[['ecDNA_statusecDNA+','Pr(>|z|)']])))
print(paste("Hazard ratio amplification:",round(exp(mb1$coefficients[['amp_statusamp.']]),2),"; p=",(summary(mb1)$coefficients[['amp_statusamp.','Pr(>|z|)']])))

In [ ]:
mb1_zph = cox.zph(mb1)
mb1_zph
ggcoxzph(mb1_zph)

### ecDNA hazard is front-loaded (24-month landmark split)

`cox.zph` above flags non-proportional hazards for `ecDNA_status`: the excess hazard is concentrated early (the amplicon-class KM shows the ecDNA curve dropping steeply through ~2 years, dashed line at 24 months, then running parallel to the others). Splitting follow-up at 24 months lets the ecDNA HR differ between periods, coded as two period-specific ecDNA indicators (`ecDNA_0_24`, `ecDNA_24p`), each the ecDNA+ vs ecDNA- effect within its window; all other covariates match `mb1`. The 24mo+ estimate rests on very few ecDNA+ events, so read it as "no detectable late effect", not "protective".

In [ ]:
# Split follow-up at 24 months; each subject contributes person-time to the interval(s) it survives into.
# Cases with events at 24 go into the (0,24] bucket.
sp <- survSplit(Surv(OS_months_5y, OS_status_5y) ~ ., data = dmb, cut = 24, episode = "period")
sp$period     <- factor(sp$period, labels = c("0-24mo", "24mo+"))
sp$ecDNA_0_24 <- as.integer(sp$ecDNA_status == "ecDNA+" & sp$period == "0-24mo")   # period-specific
sp$ecDNA_24p  <- as.integer(sp$ecDNA_status == "ecDNA+" & sp$period == "24mo+")    # ecDNA indicators (0/1)

mb1_split <- coxph(Surv(tstart, OS_months_5y, OS_status_5y) ~ ecDNA_0_24 + ecDNA_24p +
                   amp_status + sex + age_at_diagnosis + cancer_subclass, data = sp)

# period-specific ecDNA HR (cf. the whole-period HR of ~2.79 from mb1)
cis <- summary(mb1_split)$conf.int; ps <- summary(mb1_split)$coefficients[, "Pr(>|z|)"]
lab <- c(ecDNA_0_24 = "0-24mo", ecDNA_24p = "24mo+")
for (r in names(lab))
  cat(sprintf("ecDNA HR %-8s: %.2f (%.2f-%.2f), p = %.3f\n",
              lab[r], cis[r, "exp(coef)"], cis[r, "lower .95"], cis[r, "upper .95"], ps[r]))

# events behind each period x ecDNA cell (the 24mo+ ecDNA+ estimate rests on very few)
sp %>% group_by(period, ecDNA_status) %>%
  summarise(at_risk = n(), events = sum(OS_status_5y == 1), .groups = "drop")

# did splitting absorb the ecDNA PH violation?
cox.zph(mb1_split)

In [ ]:
# Forest plot of the split model. fit$n counts person-time rows, so pass the patient-level n.
vars  <- all.vars(formula(mb1_split))
n_pat <- n_distinct(sp[complete.cases(sp[, intersect(vars, names(sp))]), ]$patient_id)

p <- plotting$forest_coxph(mb1_split, n = n_pat,
  var_labels = c(ecDNA_0_24 = "ecDNA (0-24 mo)", ecDNA_24p = "ecDNA (24 mo+)",
                 amp_status = "Amplified", sex = "Sex",
                 age_at_diagnosis = "Age at diagnosis", cancer_subclass = "Subgroup"))
print(p)
plotting$save_forestploter(p, 'out/cox_mb_split_forest.png')
plotting$save_forestploter(p, 'out/cox_mb_split_forest.svg')

### KM of amplified cases stratified by ecDNA+/-, MYC family amp +/-
KM stratified by MYC family amp. log-rank tests:
```
            chr+ MYC+ chr+ MYC- ecDNA+ MYC+
chr+ MYC-   0.490     -         -          
ecDNA+ MYC+ 0.490     5.2e-06   -          
ecDNA+ MYC- 0.825     0.490     0.011
```

In [ ]:
gene_df = dplyr::bind_rows(
    load_chapman_2023_genes('../../data/external/Chapman2023/grch37_gene_list.tsv'),
    load_genes(ST_PATH)
)
dmb2 <- annotate_gene_amplifications(dmb,gene_df,c('MYC','MYCN','MYCL1')) %>%
    mutate(MYC_family_amp = replace_na(MYC | MYCN | MYCL1, FALSE) %>% factor)

In [ ]:
# KM of MYC_family_amp x ecDNA
data_subset <- dmb2 %>% 
    filter(amp_status == "amp.") %>%
    mutate(group = case_when(
        ecDNA_status == "ecDNA-" & MYC_family_amp == FALSE ~ "chr+ MYC-",
        ecDNA_status == "ecDNA-" & MYC_family_amp == TRUE ~ "chr+ MYC+",
        ecDNA_status == "ecDNA+" & MYC_family_amp == FALSE ~ "ecDNA+ MYC-",
        ecDNA_status == "ecDNA+" & MYC_family_amp == TRUE ~ "ecDNA+ MYC+"
    ))
formula = Surv(OS_months_5y, OS_status_5y) ~ group
km = survfit2(formula=formula, data = data_subset )
plt <- plotting$km_plot(km)
plt
plotting$save_ggplot("km_mbl_ec_x_mycf")
logrank <- pairwise_survdiff(formula,data_subset,p.adjust.method="BH",rho=0)
logrank
names(km$strata)

### fully parameterized Cox model including MYC family amp

In [ ]:
#dmb2 %>% head
mb2 <- coxph(Surv(OS_months_5y, OS_status_5y) ~ ecDNA_status + amp_status + MYC_family_amp+ sex + age_at_diagnosis + cancer_subclass , data = dmb2)
mb2
#plotting$cox_plot(mb2, dmb2)
#plotting$save_ggplot("cox_mbl_ec_x_mycf", width = 6, height = 5)
p <- plotting$forest_coxph(mb2)
p
plotting$save_forestploter(p,'out/cox_mbl_ec_mycf.png')
plotting$save_forestploter(p,'out/cox_mbl_ec_mycf.svg')

In [ ]:
# significant non-proportional hazards for ecDNA, MYC_family_amp.
# schoenfeld residuals stratify into cross-table of these 2 variables. underpowered.

mb2_zph = cox.zph(mb2)
mb2_zph
ggcoxzph(mb2_zph)

sr <- mb2_zph$y; tt <- mb2_zph$time
for (term in c("ecDNA_status")) {
  o <- order(abs(sr[, term]), decreasing = TRUE)[1:10]
  cat("\n", term, "\n"); print(data.frame(time = round(tt[o], 2), resid = round(sr[o, term], 1)))
}

vars <- all.vars(formula(mb3))
used <- dmb3[complete.cases(dmb3[, vars]), ]
dfb  <- residuals(mb3, type = "dfbeta")
colnames(dfb) <- names(coef(mb3))
stopifnot(nrow(dfb) == nrow(used))
used$dfb_int <- dfb[, "ecDNA_statusecDNA+"]
used[order(-abs(used$dfb_int)),
     c("patient_id","OS_months_5y","OS_status_5y","ecDNA_status","MYC_family_amp","cancer_subclass","dfb_int")] |> head(20)

### Cox model as above, but including an interaction between ecDNA x MYC family amp.

In [ ]:
# Interaction term
dmb3 <- dmb2 %>% 
    mutate(
        MYCf_x_ecDNA = ifelse(MYC_family_amp == TRUE & ecDNA_status == 'ecDNA+',1,0)
    )
mb3 <- coxph(Surv(OS_months_5y, OS_status_5y) ~ ecDNA_status + amp_status + MYC_family_amp + MYCf_x_ecDNA + sex + age_at_diagnosis + cancer_subclass , 
             data = dmb3)
mb3
#plotting$cox_plot(mb3, dmb3)
p <- plotting$forest_coxph(mb3)
p
plotting$save_forestploter(p,'out/cox_mbl_ec_x_mycf.png')
plotting$save_forestploter(p,'out/cox_mbl_ec_x_mycf.svg')

In [ ]:
# check proportional hazards assumptions.
# MYC_family_amp and MYCf_x_ecDNA fail cox.zph (proportional hazards assumption) due to 2 events for cases in
#  the rare class ecDNA- MYCf_amp+. Conclusion: underpowered.

mb3_zph = cox.zph(mb3)
mb3_zph
ggcoxzph(mb3_zph)

sr <- mb3_zph$y; tt <- mb3_zph$time
for (term in c("MYC_family_amp", "MYCf_x_ecDNA")) {
  o <- order(abs(sr[, term]), decreasing = TRUE)[1:3]
  cat("\n", term, "\n"); print(data.frame(time = round(tt[o], 2), resid = round(sr[o, term], 1)))
}

vars <- all.vars(formula(mb3))
used <- dmb3[complete.cases(dmb3[, vars]), ]
dfb  <- residuals(mb3, type = "dfbeta")
colnames(dfb) <- names(coef(mb3))
stopifnot(nrow(dfb) == nrow(used))
used$dfb_int <- dfb[, "MYCf_x_ecDNA"]
used[order(-abs(used$dfb_int)),
     c("patient_id","OS_months_5y","OS_status_5y","ecDNA_status","MYC_family_amp","cancer_subclass","dfb_int")] |> head(6)

### deprecated:
- fully parameterized cox regression on amplicon class

In [ ]:
# fully parameterized cox regression on amplicon class 
dmb1.5 <- dmb %>% mutate(amplicon_class = relevel(dmb$amplicon_class, ref = 'no amplification') )
mb1.5 <- coxph(Surv(OS_months_5y, OS_status_5y) ~ amplicon_class + sex + age_at_diagnosis + cancer_subclass, data = dmb1.5)
mb1.5
plotting$cox_plot(mb1.5, dmb1.5)
plotting$save_ggplot("cox_mb_forest_3cat", width = 6, height = 4.5)